# master

In [1]:
%reset -f

In [2]:
from pynq import PL
from pynq import (allocate, Overlay)
import numpy as np
from PIL import Image

PL.reset()

In [3]:
ol = Overlay('master-zcu102.bit')

In [4]:
help(ol)

Help on Overlay in module pynq.overlay:

<pynq.overlay.Overlay object>
    Default documentation for overlay master-zcu102.bit. The following
    attributes are available on this overlay:
    
    IP Blocks
    ----------
    zynq_ultra_ps_e_0    : pynq.overlay.DefaultIP
    
    Hierarchies
    -----------
    None
    
    Interrupts
    ----------
    None
    
    GPIO Outputs
    ------------
    None
    
    Memories
    ------------
    axi_chip2chip_0      : Memory
    PSDDR                : Memory



In [5]:
c2c = ol.axi_chip2chip_0

In [6]:
type(c2c)

pynq.pl_server.embedded_device.EmbeddedXrtMemory

In [7]:
help(c2c)

Help on EmbeddedXrtMemory in module pynq.pl_server.embedded_device object:

class EmbeddedXrtMemory(pynq.pl_server.xrt_device.XrtMemory)
 |  EmbeddedXrtMemory(device, desc)
 |  
 |  Method resolution order:
 |      EmbeddedXrtMemory
 |      pynq.pl_server.xrt_device.XrtMemory
 |      builtins.object
 |  
 |  Methods defined here:
 |  
 |  __init__(self, device, desc)
 |      Initialize self.  See help(type(self)) for accurate signature.
 |  
 |  read(self, address)
 |  
 |  write(self, address, value)
 |  
 |  ----------------------------------------------------------------------
 |  Readonly properties defined here:
 |  
 |  mmio
 |  
 |  ----------------------------------------------------------------------
 |  Methods inherited from pynq.pl_server.xrt_device.XrtMemory:
 |  
 |  __eq__(self, other)
 |      Return self==value.
 |  
 |  __hash__(self)
 |      Return hash(self).
 |  
 |  allocate(self, shape, dtype, **kwargs)
 |      Create a new  buffer in the memory bank
 |      
 |  

In [8]:
from pynq import MMIO
import time

# Define addresses
C2C_BASE_ADDR_INIT              = 0xA0000000  # Replace with actual address
C2C_BASE_ADDR_INPUT_VALUE       = 0xA0000010
C2C_BASE_ADDR_OUTPUT_OFFSET     = 0xA0000018
C2C_BASE_ADDR_OUTPUT_VALUE      = 0xFFFC0004

# Register offsets
XEXECUTE_CFG_PORT1_ADDR_DATA_PORT_DATA = 0x28

# Initialize MMIO (assuming 4KB mapping region size)
mmio_c2c = MMIO(C2C_BASE_ADDR_INIT, 0x100)
mmio_output = MMIO(C2C_BASE_ADDR_OUTPUT_VALUE, 0x100)

# Set values
init_value = 0x00000003
test_value = 0x00000042

# Write address low to DATA_PORT_DATA
addr_low = C2C_BASE_ADDR_OUTPUT_VALUE & 0xFFFFFFFF
mmio_c2c.write(XEXECUTE_CFG_PORT1_ADDR_DATA_PORT_DATA, addr_low)

# Confirm address write
read_back = mmio_c2c.read(XEXECUTE_CFG_PORT1_ADDR_DATA_PORT_DATA)
if read_back != addr_low:
    print("Bad address low!!!!")
print(f"0x{read_back:08X}")

# Test output memory mapping
mmio_output.write(0, 0xDEADBEEF)
if mmio_output.read(0) != 0xDEADBEEF:
    print("Wrong memory map!!!!!!!!!!")
else:
    print("Successfully write/read memory!")

# Write test input value
print("Trying to write input...")
mmio_input = MMIO(C2C_BASE_ADDR_INPUT_VALUE, 0x100)
mmio_input.write(0, test_value)
_ = mmio_input.read(0)  # Dummy read
time.sleep(0.01)        # Small delay
confirm_write = mmio_input.read(0)
print(f"Confirm write val: 0x{confirm_write:08X}")

# Write init value
print("Trying to initialise...")
mmio_c2c.write(0, init_value)
_ = mmio_c2c.read(0)  # Dummy read
time.sleep(0.01)      # Small delay
confirm_init = mmio_c2c.read(0)
print(f"Confirm init val: 0x{confirm_init:08X}")

# Read back result
read_value = mmio_output.read(0)
print(f"HLS Output = 0x{read_value:08X}")
if read_value == (0x40 + test_value):
    print("You rock! :)")
else:
    print("You suck! :(")


0xFFFC0004
Successfully write/read memory!
Trying to write input...
Confirm write val: 0x00000042
Trying to initialise...
Confirm init val: 0x00000004
HLS Output = 0x00000082
You rock! :)


# --------------------------

In [5]:
img2axis = ol.img2axis_0

In [6]:
# help(img2axis.register_map)

In [7]:
def start_img_to_axis(ip,buffer, eos,frame_cnt):
    
# Configure registers:
   #c2c.write(img_2_axis_base,1)
   # ip.register_map.CTRL.AP_START=1

In [8]:
def unpack_frame(packed):
    H, W_packed,ch = packed.shape  # (480, 160,4)
    W = W_packed * ch            # Unpacked width = 640
    output_tensor = np.reshape(frame, (H, W, 1))
    return output_tensor


In [9]:
def save_img(fname, tensor):
    img_tensor = np.squeeze(tensor,axis=2)  # Remove batch dim → [C, H, W]
    img = Image.fromarray(img_tensor.astype(np.uint8), mode='L')  # 'L' = 8-bit pixels, black and white
    save_path=f"{fname}.png"
    img.save(save_path)
    return save_path

In [40]:
import socket
import numpy as np
import time  # for simulating delay between frames

def send_frame(unpacked_frame, DEST_IP = '192.168.100.119',DEST_PORT = 5005):

    
    # Remove the singleton channel dimension (shape becomes 480x640)
    frame_bytes = unpacked_frame.squeeze(axis=2).tobytes()

    # Create a TCP socket
    sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    sock.connect((DEST_IP, DEST_PORT))

    # Optionally, send the frame size first (so the receiver knows what to expect)
    frame_size = len(frame_bytes)
    sock.sendall(frame_size.to_bytes(4, byteorder='big'))  # Send 4-byte length

    # Send the frame
    sock.sendall(frame_bytes)

    # Close the connection
    sock.close()

    print("Frame sent successfully.")

In [35]:
#img_to_axis(ol.img2axis_0,buff_o,True,4)
#start_img_to_axis

In [38]:
unpacked_frame = unpack_frame(frame)
print(f"type(unpacked_frame)={type(unpacked_frame)},\nunpacked_frame.shape={unpacked_frame.shape},\nunpacked_frame.dtype={unpacked_frame.dtype}")

type(unpacked_frame)=<class 'pynq.buffer.PynqBuffer'>,
unpacked_frame.shape=(480, 640, 1),
unpacked_frame.dtype=uint8


In [73]:
send_frame(unpacked_frame)

Frame sent successfully.


In [45]:
from datetime import datetime

timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
#fname=f"{timestamp}-image-output"
fname=f"image-output"
save_img(fname=fname, tensor=unpacked_frame)

'image-output.png'

# playground

In [ ]:
hasattr(ol.axi_vdma_0, 'write')  # should return True


In [ ]:
img_to_axis(ol.img2axis_0,buff_o,True,88)

In [ ]:
help(VideoMode)

In [ ]:
dir(ol.axi_vdma_0)

In [ ]:
ol.axi_vdma_0.framecount

In [ ]:
len(ol.axi_vdma_0.readchannel._frames)